# 07 — Phase 2 Clean Forward Confirmation

Runs the two hash-locked Phase 2 candidates only on post-anchor Future-OOS predictions. This notebook performs zero fitting, zero strategy selection, and no automatic promotion. Run Notebook 05 first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE = '/content/drive/MyDrive/yeniBot'
REPORT_DIR = f'{DRIVE_BASE}/reports'
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'codex/phase2-sandbox')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Repository commit:', repo_commit)
print('After a changed checkout, use Runtime -> Restart session before trusting imports.')

In [ ]:
!pip install -q "pandas>=2.2" "pyarrow>=15.0"

In [ ]:
PHASE1_RUN_ID = None  # None selects the newest report with frozen Future-OOS predictions.
CANDIDATE_ROLE = 'all'  # Keep both locked candidates; do not select on this window.
print('Fit operations: 0. Selection operations: 0.')

In [ ]:
import json
import shutil
import time
from pathlib import Path

from yenibot.automation.phase2_forward import main as run_phase2_forward

experiment_reports = Path(REPORT_DIR) / 'experiments'
if PHASE1_RUN_ID:
    report_dir = experiment_reports / str(PHASE1_RUN_ID)
else:
    eligible = sorted(
        (
            path for path in experiment_reports.iterdir()
            if path.is_dir()
            and (path / 'frozen_candidate_manifest.json').exists()
            and (path / 'future_oos_predictions.parquet').exists()
        ),
        key=lambda path: path.name,
    )
    if not eligible:
        raise FileNotFoundError('No Phase 1 report contains frozen Future-OOS predictions. Run Notebook 05 first.')
    report_dir = eligible[-1]

local_output = Path('/content') / f'phase2_forward_{report_dir.name}'
args = [
    '--input-dir', str(report_dir),
    '--report-dir', str(report_dir),
    '--output-dir', str(local_output),
    '--candidate-role', CANDIDATE_ROLE,
]
started = time.perf_counter()
run_phase2_forward(args)
elapsed = time.perf_counter() - started
decision_path = local_output / 'phase2_forward_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print('Phase 1 source:', report_dir)
print('Forward status:', decision['status'])
print('Accepted signals:', decision['boundary']['accepted_signal_count'])
print('Coverage days:', decision['boundary']['coverage_days'])
print(f'Runtime: {elapsed:.2f} seconds')
print('Fit operations performed:', decision['fit_operations_performed'])
print('Selection operations performed:', decision['selection_operations_performed'])

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for item in decision.get('candidate_results', []):
    rows.append({
        'role': item['role'],
        'strategy_id': item['strategy_id'],
        'status': item['gate']['status'],
        'trades': item['base']['trade_count'],
        'base_return': item['base']['compounded_return'],
        'adverse_return': item['adverse']['compounded_return'],
        'profit_factor': item['base']['profit_factor'],
        'max_drawdown': item['base']['max_drawdown'],
        'failed_checks': ', '.join(item['gate']['failed_checks']),
    })
display(pd.DataFrame(rows))
if not rows:
    print('No post-anchor predictions yet. This is a valid waiting state, not an error.')

local_bundle = Path(shutil.make_archive(
    str(Path('/content') / f'phase2_forward_bundle_{report_dir.name}'),
    'zip',
    root_dir=local_output.parent,
    base_dir=local_output.name,
))

def durable_copy(source, destination, attempts=5):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, attempts + 1):
        temporary = destination.with_name(destination.name + '.tmp')
        try:
            shutil.copyfile(source, temporary)
            os.replace(temporary, destination)
            return destination
        except OSError:
            if temporary.exists():
                temporary.unlink(missing_ok=True)
            if attempt == attempts:
                raise
            time.sleep(float(attempt))

versioned = durable_copy(local_bundle, Path(REPORT_DIR) / f'phase2_forward_bundle_{report_dir.name}.zip')
latest = durable_copy(local_bundle, Path(REPORT_DIR) / 'phase2_latest_forward_bundle.zip')
print('Versioned forward bundle:', versioned)
print('Latest forward bundle:', latest)

In [ ]:
from google.colab import runtime
print('Forward confirmation finished. Releasing the Colab runtime.')
runtime.unassign()